# Could we even have spotted a drop in likes-per-view if there was one?

**Data used:** the experiment itself (who got a CN reply + Views/Likes/Shares)

**Short answer:** Barely. We’d need ~2.8x more tweets for likes and ~6x for shares. So "no effect" here means "too small to catch", not "definitely nothing".




# Raw-Data Extension — Mean-based Power on Engagement-Rate Change

**Question.** The main analysis (`main_effect.ipynb` section 3.4.3 / 3.4.3b) tracked the **rate change** for Likes/Views and Shares/Views (Day 13 rate − Day 0 rate) but only as a descriptive plot. Would a larger sample have produced a significant **mean-based** difference between Treatment and Control?

**Method.** Bootstrap-resample both groups at multipliers 1×–10× our N (1,000 trials per multiplier), run a two-sided Welch t-test per trial, compute empirical power.

**Outcomes (matching main notebook):**
- `likes_rate_change = Likes_Day13 / (Views_Day13 + 1) − Likes_Day0 / (Views_Day0 + 1)`
- `shares_rate_change = Shares_Day13 / (Views_Day13 + 1) − Shares_Day0 / (Views_Day0 + 1)`


In [ ]:
import os, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings('ignore')
np.random.seed(42)

BASE = Path.cwd()
while not (BASE / 'data').is_dir() and BASE != BASE.parent:
    BASE = BASE.parent
DATA = BASE / 'data'
OUT  = BASE / 'outputs' / 'engagement_rate_power'
OUT.mkdir(parents=True, exist_ok=True)

CONTROL_XL    = DATA / 'Control_Group.xlsx'
TREATMENT_XL  = DATA / 'Treatment_Group.xlsx'
MONITORING_XL = DATA / 'Tweet Monitoring.xlsx'

MULTIPLIERS = [1.0, 1.5, 2.0, 3.0, 5.0, 10.0]
N_TRIALS    = 1000
print(f'Out: {OUT}')


In [ ]:
# Domain-aware date parser (Dec=2025, Jan=2026)+
def parse_mixed_date(date_val):
    if pd.isna(date_val): return pd.NaT
    s = str(date_val).strip()
    def is_valid(year, month):
        return (year == 2025 and month == 12) or (year == 2026 and month == 1)
    if '-' in s and s[:4].isdigit():
        parts = s.split('-')
        year = int(parts[0]); n1 = int(parts[1]); n2 = int(parts[2].split()[0])
        if is_valid(year, n1):    month, day = n1, n2
        elif is_valid(year, n2):  month, day = n2, n1
        else:                      month, day = n1, n2
        return pd.Timestamp(year=year, month=month, day=day)
    elif '/' in s:
        parts = s.split('/')
        n1 = int(parts[0]); n2 = int(parts[1]); year = int(parts[2].split()[0])
        if is_valid(year, n2):    day, month = n1, n2
        elif is_valid(year, n1):  month, day = n1, n2
        else:                      day, month = n1, n2
        return pd.Timestamp(year=year, month=month, day=day)
    return pd.to_datetime(date_val, errors='coerce')
print('Date parser ready.')


## Build outcomes — `likes_rate_change` and `shares_rate_change` per tweet

In [ ]:
control   = pd.read_excel(CONTROL_XL)
treatment = pd.read_excel(TREATMENT_XL)
monitoring = pd.read_excel(MONITORING_XL)
monitoring['Sample_Date']           = monitoring['Sample_Date'].apply(parse_mixed_date)
monitoring['Start_Date (Creation)'] = monitoring['Start_Date (Creation)'].apply(parse_mixed_date)
monitoring['Day'] = (monitoring['Sample_Date'] - monitoring['Start_Date (Creation)']).dt.days

pivot = monitoring.pivot_table(index='URL', columns='Day',
                               values=['Views','Likes','Shares'], aggfunc='first')
pivot.columns = [f'{m}_Day{d}' for m, d in pivot.columns]
pivot = pivot.reset_index()

group_map = pd.concat([control[['URL']].assign(Group='Control'),
                       treatment[['URL']].assign(Group='Treatment')])
df = pivot.merge(group_map, on='URL', how='left')

# Day0 fill for Likes/Shares (convention from main + later notebooks)
for m in ['Likes', 'Shares']:
    df[f'{m}_Day0'] = df[f'{m}_Day0'].fillna(0)

# Per-day rate then change (Day 13 rate - Day 0 rate). Use (Views + 1) per main convention.
df['like_rate_d0']  = df['Likes_Day0']  / (df['Views_Day0']  + 1)
df['like_rate_d13'] = df['Likes_Day13'] / (df['Views_Day13'] + 1)
df['likes_rate_change'] = df['like_rate_d13'] - df['like_rate_d0']

df['share_rate_d0']  = df['Shares_Day0']  / (df['Views_Day0']  + 1)
df['share_rate_d13'] = df['Shares_Day13'] / (df['Views_Day13'] + 1)
df['shares_rate_change'] = df['share_rate_d13'] - df['share_rate_d0']

print(f'df: {df.shape}')
print()
# Sanity: direction at observed N — should match main notebook 3.4.3b (Treatment slightly lower quality)
for rate in ['likes_rate_change', 'shares_rate_change']:
    t = df.loc[df['Group']=='Treatment', rate].dropna().values
    c = df.loc[df['Group']=='Control',   rate].dropna().values
    tstat, pval = stats.ttest_ind(t, c, equal_var=False)
    print(f'  {rate}:  mean_T={t.mean():+.5f}  mean_C={c.mean():+.5f}  diff={t.mean()-c.mean():+.5f}  '
          f'Welch t={tstat:+.3f}  p={pval:.4f}  (n_T={len(t)}, n_C={len(c)})')


## Bootstrap power simulation (mean-based, Welch t)

In [ ]:
def bootstrap_power(ctrl_vals, trt_vals, k, n_trials=1000, alpha=0.05, seed=None):
    rng = np.random.default_rng(seed)
    n_c = int(round(len(ctrl_vals) * k))
    n_t = int(round(len(trt_vals)  * k))
    sigs = 0
    for _ in range(n_trials):
        cs = rng.choice(ctrl_vals, size=n_c, replace=True)
        ts = rng.choice(trt_vals,  size=n_t, replace=True)
        try:
            _, p = stats.ttest_ind(ts, cs, equal_var=False)
            if p < alpha: sigs += 1
        except ValueError:
            pass
    return sigs / n_trials, n_c, n_t


rows = []
import time
t0 = time.time()
for rate in ['likes_rate_change', 'shares_rate_change']:
    t_vals = df.loc[df['Group']=='Treatment', rate].dropna().values
    c_vals = df.loc[df['Group']=='Control',   rate].dropna().values
    print(f'\n=== {rate}  (n_C={len(c_vals)}, n_T={len(t_vals)}) ===')
    for k in MULTIPLIERS:
        seed = 42 + int(k*1000) + hash(rate) % 10_000
        power, n_c, n_t = bootstrap_power(c_vals, t_vals, k=k, n_trials=N_TRIALS, seed=seed)
        print(f'  k={k:>4}×  N={n_c:>5}/{n_t:>5}  power={power:.3f}')
        rows.append(dict(rate=rate, multiplier=k, n_ctrl=n_c, n_trt=n_t, power=power))

print(f'\nTotal elapsed: {time.time()-t0:.1f}s')
power_df = pd.DataFrame(rows)
power_df.to_csv(OUT / 'rate_change_power_curves.csv', index=False)
print(f'Saved: rate_change_power_curves.csv')


## N for 80% power + verdict

In [ ]:
def n_for_80(sub, target=0.80):
    sub = sub.sort_values('multiplier').reset_index(drop=True)
    for i in range(len(sub) - 1):
        p_lo, p_hi = sub.loc[i,'power'], sub.loc[i+1,'power']
        n_lo = (sub.loc[i,'n_ctrl']   + sub.loc[i,'n_trt'])   / 2
        n_hi = (sub.loc[i+1,'n_ctrl'] + sub.loc[i+1,'n_trt']) / 2
        if p_lo >= target: return float(n_lo)
        if p_hi >= target:
            return float(n_lo + (target - p_lo) / (p_hi - p_lo) * (n_hi - n_lo))
    return None

verdicts = []
for rate in ['likes_rate_change', 'shares_rate_change']:
    sub = power_df[power_df['rate']==rate]
    n80 = n_for_80(sub, 0.80)
    obs_p = float(sub.loc[sub['multiplier']==1.0, 'power'].iloc[0])
    n_orig = float((sub.loc[sub['multiplier']==1.0,'n_ctrl'].iloc[0] +
                    sub.loc[sub['multiplier']==1.0,'n_trt'].iloc[0]) / 2)
    verdicts.append(dict(
        rate=rate,
        observed_power=obs_p,
        n_orig_per_group=n_orig,
        n_for_80=n80 if n80 else np.nan,
        multiplier_for_80=(n80 / n_orig) if n80 else np.nan,
        verdict=('reaches 80%' if n80 else '>10× orig N, likely truly null'),
    ))
v_df = pd.DataFrame(verdicts)
v_df.to_csv(OUT / 'rate_change_n_for_80.csv', index=False)
print(v_df.to_string(index=False))


## Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.0), dpi=150)
colors = {'likes_rate_change': '#d62728', 'shares_rate_change': '#2ca02c'}
labels = {'likes_rate_change': 'Likes-rate change',
          'shares_rate_change': 'Shares-rate change'}
for rate in ['likes_rate_change', 'shares_rate_change']:
    sub = power_df[power_df['rate']==rate].sort_values('n_trt')
    n_avg = (sub['n_ctrl'] + sub['n_trt']) / 2
    ax.plot(n_avg, sub['power'], '-o', color=colors[rate], lw=2, markersize=7, label=labels[rate])
ax.axhline(0.80, color='black', lw=1, ls='--', alpha=0.5)
ax.text(power_df['n_ctrl'].max()*0.95, 0.81, '80% power', ha='right', va='bottom',
        fontsize=9, color='black', alpha=0.7)
ax.axhline(0.05, color='gray', lw=0.7, ls=':', alpha=0.5)
ax.set_xscale('log')
ax.set_xlabel('Sample size per group (log scale)')
ax.set_ylabel('Empirical power (fraction of Welch-t trials with p<0.05)')
ax.set_title('Mean-based power on engagement-rate change (Day 13 rate − Day 0 rate)')
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3, linestyle='--')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(OUT / 'fig_rate_change_power.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: fig_rate_change_power.png')
